In [6]:
import pandas as pd

# ── 0. Charger les données brutes ──
df = pd.read_csv("../data/activities.csv")

# ── 1. Garder uniquement les courses à pied ──
df = df[df["sport_type"] == "Run"].copy()
print(f"Runs only: {len(df)} activities")

# ── 2. Convertir les dates ──
df["start_date_local"] = pd.to_datetime(df["start_date_local"], utc=True)

# ── 3. Ajouter des colonnes utiles ──
jours_fr = {
    "Monday": "Lundi", "Tuesday": "Mardi", "Wednesday": "Mercredi",
    "Thursday": "Jeudi", "Friday": "Vendredi",
    "Saturday": "Samedi", "Sunday": "Dimanche"
}
mois_fr = {
    1: "janvier", 2: "février", 3: "mars", 4: "avril",
    5: "mai", 6: "juin", 7: "juillet", 8: "août",
    9: "septembre", 10: "octobre", 11: "novembre", 12: "décembre"
}

df["day_of_week"] = df["start_date_local"].dt.day_name().map(jours_fr)
df["date_fr"]     = df["start_date_local"].apply(
    lambda x: f"{x.day} {mois_fr[x.month]} {x.year}"
)
df["hour_fr"]     = df["start_date_local"].apply(
    lambda x: f"{x.hour}h{x.minute:02d}"
)
df["date"]        = df["start_date_local"].dt.date
df["hour"]        = df["start_date_local"].dt.hour
df["month"]       = df["start_date_local"].dt.tz_localize(None).dt.to_period("M")
df["year"]        = df["start_date_local"].dt.year

# ── 4. Convertir les unités ──
df["distance_km"]   = df["distance_m"] / 1000
df["pace_min_km"]   = (1 / df["average_speed_m_s"]) / 60 * 1000
df["pause_seconds"] = df["elapsed_time_seconds"] - df["moving_time_seconds"]

# ── 5. Sauvegarder df_map avant de dropper (pour la carte) ──
df_map = df[["id", "map_id", "summary_polyline",
             "start_latlng", "end_latlng",
             "location_city", "date"]].copy()

# ── 6. Supprimer les colonnes inutiles pour l'analyse ──
cols_to_drop = [
    "external_id", "upload_id", "type", "start_date",
    "timezone", "workout_type", "private", "visibility",
    "commute", "trainer", "utc_offset", "has_heartrate",
    "average_cadence",
    "start_latlng", "end_latlng", "map_id",
    "summary_polyline", "gear_id",
    "location_city", "location_state", "location_country",
    "distance_m",
    "elapsed_time_seconds",
    "average_watts", "max_watts", "weighted_average_watts",
    "device_watts", "kilojoules",
    "kudos_count", "comment_count", "photo_count",
    "total_photo_count", "achievement_count",
]
df = df.drop(columns=cols_to_drop)

print(f"Colonnes restantes : {list(df.columns)}")
print(f"Shape : {df.shape}")

# ── 7. Sauvegarder ──
df.to_csv("../data/activities_clean.csv", index=False)
df_map.to_csv("../data/activities_map.csv", index=False)

print("Activities saved to data/activities_clean.csv and data/activities_map.csv")


Runs only: 112 activities
Colonnes restantes : ['id', 'name', 'sport_type', 'manual', 'flagged', 'start_date_local', 'moving_time_seconds', 'average_speed_m_s', 'max_speed_m_s', 'total_elevation_gain_m', 'elev_high_m', 'elev_low_m', 'average_heartrate', 'max_heartrate', 'pr_count', 'suffer_score', 'day_of_week', 'date_fr', 'hour_fr', 'date', 'hour', 'month', 'year', 'distance_km', 'pace_min_km', 'pause_seconds']
Shape : (112, 26)
Activities saved to data/activities_clean.csv and data/activities_map.csv


In [7]:
df.columns

Index(['id', 'name', 'sport_type', 'manual', 'flagged', 'start_date_local',
       'moving_time_seconds', 'average_speed_m_s', 'max_speed_m_s',
       'total_elevation_gain_m', 'elev_high_m', 'elev_low_m',
       'average_heartrate', 'max_heartrate', 'pr_count', 'suffer_score',
       'day_of_week', 'date_fr', 'hour_fr', 'date', 'hour', 'month', 'year',
       'distance_km', 'pace_min_km', 'pause_seconds'],
      dtype='object')